In [132]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from typing import List
from tqdm import tqdm
import random
import seaborn as sns
from typing import Tuple
import pandas as pd

In [133]:
df_users = pd.read_csv('KION_DATASET/data_original/users.csv')
df_interactions = pd.read_csv('KION_DATASET/interactions.csv')
df_items = pd.read_csv('KION_DATASET/data_original/items.csv')

In [134]:
df_interactions.head()

,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250.0,72.0
1,699317,1659,2021-05-29,8317.0,100.0
2,656683,7107,2021-05-09,10.0,0.0
3,864613,7638,2021-07-05,14483.0,100.0
4,964868,9506,2021-04-30,6725.0,100.0


In [135]:
df_users.head()

,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,М,1
1,962099,age_18_24,income_20_40,М,0
2,1047345,age_45_54,income_40_60,Ж,0
3,721985,age_45_54,income_20_40,Ж,0
4,704055,age_35_44,income_60_90,Ж,0


In [136]:
df_items.head()

,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."
2,10716,film,Тактическая сила,Tactical Force,2011.0,"криминал, зарубежные, триллеры, боевики, комедии",Канада,NaN,16.0,NaN,Адам П. Калтраро,"Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Профессиональный рестлер Стив Остин («Все или ...,"Тактическая, сила, 2011, Канада, бандиты, ганг..."
3,7868,film,45 лет,45 Years,2015.0,"драмы, зарубежные, мелодрамы",Великобритания,NaN,16.0,NaN,Эндрю Хэй,"Александра Риддлстон-Барретт, Джеральдин Джейм...","Шарлотта Рэмплинг, Том Кортни, Джеральдин Джей...","45, лет, 2015, Великобритания, брак, жизнь, лю..."
4,16268,film,Все решает мгновение,NaN,1978.0,"драмы, спорт, советские, мелодрамы",СССР,NaN,12.0,Ленфильм,Виктор Садовский,"Александр Абдулов, Александр Демьяненко, Алекс...",Расчетливая чаровница из советского кинохита «...,"Все, решает, мгновение, 1978, СССР, сильные, ж..."


In [137]:
df_interactions['last_watch_dt'].unique()
df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'], errors='coerce')
df_interactions = df_interactions.dropna(subset=['last_watch_dt'])

In [138]:
old_item = df_items[df_items['release_year'] > 1960.0] # уберем старые фильмы до 1960 года
df_interactions = df_interactions[df_interactions['item_id'].isin(old_item['item_id'])]
df_interactions = df_interactions.drop(['total_dur'], axis=1, inplace=False) # уберем время просмотра

In [139]:
# from datetime import datetime, timedelta
# test_size_days = 8
# size_test = 1 # в test берем последнюю неделю
# data = df_interactions[(df_interactions['watched_pct']>50.0)] # уберем фильмы с просмотром менее 25%
# items_data = data['item_id'].value_counts()
# active_items = items_data[items_data > 5].index # уберем фильмы с которыми взаимодействовали менее 5 раз
# data = data[data['item_id'].isin(active_items)]
# data_train = data[data['last_watch_dt'] < data['last_watch_dt'].max() - timedelta(days=test_size_days)]
# data_test = data[data['last_watch_dt'] >= data['last_watch_dt'].max() - timedelta(days=test_size_days)]
# # data_train = data[data['last_watch_dt'] < data['last_watch_dt'].max() - timedelta(weeks=size_test)]
# # data_test = data[data['last_watch_dt'] >= data['last_watch_dt'].max() - timedelta(weeks=size_test)]

In [140]:
from datetime import datetime, timedelta
test_size_days = 10

# Тестовый промежуток времени равен 10 дней
max_date = df_interactions['last_watch_dt'].max()
test_start = max_date - timedelta(days=test_size_days)

df_interactions = df_interactions[(df_interactions['watched_pct']>50.0)] # уберем фильмы с просмотром менее 50%
df_interactions_train = df_interactions[df_interactions['last_watch_dt'] < test_start]
df_interactions_test = df_interactions[df_interactions['last_watch_dt'] >= test_start]

# df_interactions_train = df_interactions[df_interactions['last_watch_dt'] < df_interactions['last_watch_dt'].max() - timedelta(days=test_size_days)]
# df_interactions_test = df_interactions[df_interactions['last_watch_dt'] >= df_interactions['last_watch_dt'].max() - timedelta(days=test_size_days)]

In [141]:
train_sorted = df_interactions_train.sort_values(['user_id', 'last_watch_dt'])
    
train = train_sorted.groupby('user_id').tail(15) # у пользователей в train берем последние 15 фильмов

In [142]:
test_users = set(df_interactions_test['user_id'].unique())
train_users = set(train['user_id'].unique())
common_users = test_users & train_users
    
print(f"Пользователей в test: {len(test_users)}")
print(f"Пользователей в train: {len(train_users)}")
print(f"Общих пользователей: {len(common_users)}")

Пользователей в test: 47347
Пользователей в train: 281756
Общих пользователей: 21391


In [143]:
def get_active_users_ids(df, min_interactions=2):
    """Возвращает ID пользователей, у которых больше минимального числа взаимодействий."""
    counts = df['user_id'].value_counts()
    return set(counts[counts >= min_interactions].index)

# 1. Находим активных пользователей в каждом наборе данных
active_train_ids = get_active_users_ids(train)
active_test_ids = get_active_users_ids(df_interactions_test)

# 2. Находим пересечение (пользователи, активные и в train, и в test)
common_active_users = active_train_ids & active_test_ids

# 3. Фильтруем исходные данные, оставляя только общих активных пользователей
df_interactions_train = train[train['user_id'].isin(common_active_users)]
df_interactions_test = df_interactions_test[df_interactions_test['user_id'].isin(common_active_users)]

In [144]:
# # Функция для получения индексов активных пользователей
# get_active = lambda df: df['user_id'].value_counts().loc[lambda x: x > 1].index

# # Получаем списки активных пользователей
# train_active_users = set(get_active(train))
# test_active_users = set(get_active(df_interactions_test))

# # Оставляем только пересечение пользователей в обоих датасетах
# common_users = train_active_users & test_active_users

# df_interactions_train = train[train['user_id'].isin(common_users)]
# df_interactions_test = df_interactions_test[df_interactions_test['user_id'].isin(common_users)]

In [145]:
test_users = set(df_interactions_test['user_id'].unique())
train_users = set(df_interactions_train['user_id'].unique())
common_users = test_users & train_users
    
print(f"Пользователей в test: {len(test_users)}")
print(f"Пользователей в train: {len(train_users)}")
print(f"Общих пользователей: {len(common_users)}")

Пользователей в test: 5019
Пользователей в train: 5019
Общих пользователей: 5019


In [146]:
def precision(recommended_list, bought_list):
    recommended = np.array(recommended_list)
    bought = np.array(bought_list)

    # флаги: какие рекомендованные товары действительно куплены
    flags = np.isin(recommended, bought)

    return flags.sum() / len(recommended)


def precision_at_k(recommended_list, bought_list, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)

    flags = np.isin(recommended, bought)

    return flags.sum() / k


def money_precision_at_k(recommended_list, bought_list, prices_recommended, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)
    prices = np.array(prices_recommended[:k])

    # флаги: куплен ли товар
    flags = np.isin(recommended, bought)

    # учитываем деньги
    return np.dot(flags, prices) / prices.sum()

In [147]:
def recall(recommended_list, bought_list):
    recommended = np.array(recommended_list)
    bought = np.array(bought_list)

    # какие купленные товары были среди рекомендованных
    flags = np.isin(bought, recommended)

    return flags.sum() / len(bought)


def recall_at_k(recommended_list, bought_list, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)

    flags = np.isin(bought, recommended)

    return flags.sum() / len(bought)


def money_recall_at_k(recommended_list, bought_list, prices_recommended, prices_bought, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)
    prices_bought = np.array(prices_bought)

    # флаги: купленный товар есть в топ-k рекомендациях
    flags = np.isin(bought, recommended)

    # учитываем деньги (важны цены купленных товаров)
    return np.dot(flags, prices_bought) / prices_bought.sum()

In [148]:
def ap_k(recommended_list, bought_list, k=5):
    if not recommended_list or not len(bought_list):
        return 0.0
    
    # Ограничиваем список рекомендаций сверху значением k
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)
    
    # Флаги: попал ли рекомендованный товар в список покупок
    flags = np.isin(recommended, bought)
    
    # Если нет попаданий — AP = 0
    if not np.any(flags):
        return 0.0
    
    sum_precision = 0.0
    hits = 0
    
    # Итерируем только по фактической длине рекомендаций (но не более k)
    for i, is_hit in enumerate(flags):
        if is_hit:
            hits += 1
            precision_at_i = hits / (i + 1)  # Precision@i+1
            sum_precision += precision_at_i
    
    # Нормализуем на минимальное из (k, число релевантных)
    normalization = min(k, len(bought))
    return sum_precision / normalization if normalization > 0 else 0.0


def map_k(recommended_list, bought_list, k=5, u=1):
    
    # your_code
    if u == 1:
        return ap_k(recommended_list[u-1], bought_list[u-1], k=5)
    
    sum = 0
    for i in range(0, u):
        ap_k_map = ap_k(recommended_list[i], bought_list[i], k=5)
        sum += ap_k_map

    result = sum / u
    
    return result

In [149]:
data_test_group = df_interactions_test.groupby(df_interactions_test['user_id'])['item_id'].unique().reset_index()
data_test_group.columns = ['user_id', 'item_id']
data_test_group

,user_id,item_id
0,259,"[10509, 10772]"
1,655,"[11188, 5199]"
2,753,"[9327, 8350]"
3,835,"[5434, 11640, 10878]"
4,960,"[8636, 12770, 6809]"
...,...,...
5014,1096841,"[10219, 3620]"
5015,1097296,"[3006, 16087]"
5016,1097444,"[13650, 12841, 13099, 1124, 12250, 2483]"
5017,1097459,"[11844, 7793]"


In [150]:
userids = data_test_group['user_id'].values
 
userids_test = np.arange(len(userids))

In [151]:
# users_unique_test = data_test_group['user_id']
# items_unique = df_interactions['item_id'].unique()

# size_recommend = 10

# recommendations = []
# for user in users_unique_test:
#     random_items = np.random.choice(items_unique, size=size_recommend, replace=False)
#     recommendations.append(random_items)

# result_random = data_test_group.copy()
# result_random['random_recommendation'] = recommendations

Random

In [152]:
def generate_random_recommendations(df_users, df_items, top_n=10, seed=42):
    """
    Генерирует случайные рекомендации для уникальных пользователей.
    """
    np.random.seed(seed)
    
    # 1. Получаем уникальных пользователей и все доступные товары
    unique_users = df_users['user_id'].unique()
    all_items = df_items['item_id'].unique()
    
    # 2. Защита: если товаров меньше, чем нужно рекомендовать
    k = min(top_n, len(all_items))
    
    # 3. Генерируем рекомендации в виде словаря {user_id: [items]}
    # Это быстрее и безопаснее, чем цикл с append
    recommendations_map = {
        user: np.random.choice(all_items, size=k, replace=False).tolist()
        for user in unique_users
    }
    
    # 4. Создаем копию датафрейма и мапим рекомендации по user_id
    result = df_users.copy()
    result['random_recommendation'] = result['user_id'].map(recommendations_map)
    
    return result

# Использование
result_random = generate_random_recommendations(
    df_users=data_test_group, 
    df_items=df_interactions, 
    top_n=10
)

In [153]:
# import numpy as np

# np.random.seed(42)

# unique_users = data_test_group['user_id'].unique()
# all_items = df_interactions['item_id'].unique()
# k = min(10, len(all_items))

# # Создаем список словарей для быстрого создания DataFrame
# rec_data = []
# for user in unique_users:
#     items = np.random.choice(all_items, size=k, replace=False)
#     for item in items:
#         rec_data.append({'user_id': user, 'recommended_item': item})

# df_recommendations = pd.DataFrame(rec_data)

# # Если нужно объединить с исходным тестом (например, для слияния по user_id)
# result_random = data_test_group.merge(df_recommendations, on='user_id', how='left')

In [154]:
def getRandomRecommend(items, k=10):
  return np.random.choice(items, k)

In [155]:
def getFirstK(items, k=10):
  return items[:k]

In [156]:
# items взять из тестового набора



print('random map_k =', map_k(result_random['random_recommendation'], result_random['item_id'], k=10, u=len(result_random)))


print(map_k)

random map_k = 0.00018429966128710897
<function map_k at 0x00000237D7DADA20>


Popular

In [157]:
# def popularity(train, n=10):
#     train_group = train.groupby(train['item_id'])['user_id'].unique().reset_index()
#     train_group.columns = ['item_id', 'user_id']

#     train_group['user_sum'] = train_group['user_id'].apply(len)
#     top_10_train = train_group.nlargest(n, 'user_sum')
#     top_10 = top_10_train['item_id']

#     return top_10.tolist()

# popular = popularity(df_interactions_train, n=10)
# result_popular = data_test_group.copy()
# result_popular['popular_recommendation'] = result_popular['user_id'].apply(lambda x: popular)

In [158]:
def get_popular_items(train, n=10):
    """Возвращает список из n самых популярных товаров."""
    top_items = train['item_id'].value_counts().nlargest(n)
    return top_items.index.tolist()

# 1. Получаем список популярных товаров (длиной 10)
popular = get_popular_items(df_interactions_train, n=10)

# 2. Присваиваем рекомендации
result_popular = data_test_group.copy()

# ВАРИАНТ А: Список списков (Рекомендуемый)
# Создаем список, где каждый элемент — это копия списка popular.
# Длина этого нового списка будет равна количеству строк в датафрейме.
result_popular['popular_recommendation'] = [popular] * len(result_popular)

In [159]:


result_popular



,user_id,item_id,popular_recommendation
0,259,"[10509, 10772]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."
1,655,"[11188, 5199]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."
2,753,"[9327, 8350]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."
3,835,"[5434, 11640, 10878]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."
4,960,"[8636, 12770, 6809]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."
...,...,...,...
5014,1096841,"[10219, 3620]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."
5015,1097296,"[3006, 16087]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."
5016,1097444,"[13650, 12841, 13099, 1124, 12250, 2483]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."
5017,1097459,"[11844, 7793]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."


In [160]:


print('popular map_k =', map_k(result_popular['popular_recommendation'], result_popular['item_id'], k=10, u=len(result_popular)))



popular map_k = 0.025992727634987063


UserKNN

In [161]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

class UserKNN:
    def __init__(self, k=5, min_similarity=0.1):
        self.k = k
        self.min_similarity = min_similarity
        
        # Маппинги ID -> Индекс для быстрого доступа O(1)
        self.user_id_to_idx = {}
        self.idx_to_user_id = {}
        self.item_id_to_idx = {}
        self.idx_to_item_id = {}
        
        self.user_item_matrix = None  # Разреженная матрица (CSR)
        self.user_similarity = None   # Матрица схожести (Плотная, память ограничивает)
        self.n_users = 0
        self.n_items = 0

    def fit(self, interactions):
        """Обучение модели: построение матриц и расчет схожести."""
        self._build_mappings(interactions)
        self._build_matrix(interactions)
        self._compute_similarity()
        return self

    def _build_mappings(self, interactions):
        """Создает словари для быстрого перевода ID в индексы и обратно."""
        unique_users = interactions['user_id'].unique()
        unique_items = interactions['item_id'].unique()
        
        self.user_id_to_idx = {uid: idx for idx, uid in enumerate(unique_users)}
        self.idx_to_user_id = {idx: uid for uid, idx in self.user_id_to_idx.items()}
        
        self.item_id_to_idx = {iid: idx for idx, iid in enumerate(unique_items)}
        self.idx_to_item_id = {idx: iid for iid, idx in self.item_id_to_idx.items()}
        
        self.n_users = len(unique_users)
        self.n_items = len(unique_items)

    def _build_matrix(self, interactions):
        """Создает разреженную матрицу пользователь-товар."""
        user_codes = interactions['user_id'].map(self.user_id_to_idx)
        item_codes = interactions['item_id'].map(self.item_id_to_idx)
        
        # Удаляем строки, где маппинг не удался (если есть новые ID, хотя в fit их не должно быть)
        mask = user_codes.notna() & item_codes.notna()
        
        self.user_item_matrix = csr_matrix(
            (interactions.loc[mask, 'watched_pct'], 
             (user_codes.loc[mask], item_codes.loc[mask])),
            shape=(self.n_users, self.n_items)
        )

    def _compute_similarity(self):
        """Вычисляет матрицу косинусной схожести между пользователями."""
        # Внимание: результат плотный (N_users x N_users). 
        # Для >50k пользователей лучше использовать Approximate NN (например, FAISS/Annoy).
        self.user_similarity = cosine_similarity(self.user_item_matrix, dense_output=True)
        np.fill_diagonal(self.user_similarity, 0)

    def _get_user_idx(self, user_id):
        return self.user_id_to_idx.get(user_id)

    def _get_item_idx(self, item_id):
        return self.item_id_to_idx.get(item_id)

    def _get_similar_users(self, user_idx):
        """Возвращает индексы и веса похожих пользователей для данного индекса."""
        similarities = self.user_similarity[user_idx]
        
        # Берем топ-k + фильтр по порогу
        # argsort быстрее, чем поиск по условию + сортировка
        top_indices = np.argsort(similarities)[::-1][:self.k]
        
        similar_users = []
        for idx in top_indices:
            sim = similarities[idx]
            if sim >= self.min_similarity:
                similar_users.append((idx, sim))
            else:
                break # Так как отсортировано по убыванию, дальше смысла нет
        return similar_users

    def predict(self, user_id, item_id):
        """Предсказывает рейтинг для конкретной пары пользователь-товар."""
        user_idx = self._get_user_idx(user_id)
        item_idx = self._get_item_idx(item_id)
        
        if user_idx is None or item_idx is None:
            return 0.0

        similar_users = self._get_similar_users(user_idx)
        if not similar_users:
            return 0.0

        # Векторизованный сбор рейтингов соседей
        neighbor_indices = [idx for idx, _ in similar_users]
        neighbor_weights = np.array([w for _, w in similar_users])
        
        # Получаем рейтинги соседей на этот товар из разреженной матрицы
        # .toarray() здесь безопасно, так как берем один столбец (или срез)
        neighbor_ratings = self.user_item_matrix[neighbor_indices, item_idx].toarray().flatten()
        
        # Фильтруем соседей, которые не оценили этот товар
        mask = neighbor_ratings > 0
        if not np.any(mask):
            return 0.0
            
        weights = neighbor_weights[mask]
        ratings = neighbor_ratings[mask]
        
        if np.sum(weights) > 0:
            return float(np.average(ratings, weights=weights))
        else:
            return float(np.mean(ratings))

    def recommend(self, user_id, n_recommendations=10):
        """Генерирует топ-N рекомендаций для пользователя (Векторизовано)."""
        user_idx = self._get_user_idx(user_id)
        if user_idx is None:
            return []

        # 1. Находим похожих пользователей ОДИН РАЗ
        similar_users = self._get_similar_users(user_idx)
        if not similar_users:
            return []

        neighbor_indices = [idx for idx, _ in similar_users]
        neighbor_weights = np.array([w for _, w in similar_users])

        # 2. Получаем матрицу рейтингов соседей (K соседей x Все товары)
        # Это быстро, так как матрица разреженная
        neighbors_matrix = self.user_item_matrix[neighbor_indices] 
        
        # 3. Считаем взвешенный скор для ВСЕХ товаров сразу
        # (Weights.T @ Matrix) -> вектор скоров длиной N_items
        scores = neighbor_weights @ neighbors_matrix
        
        # 4. Обнуляем товары, которые пользователь уже видел
        user_rated_items = self.user_item_matrix[user_idx].indices
        scores[user_rated_items] = -np.inf 

        # 5. Находим топ-N индексов
        # argpartition быстрее argsort для получения только топ-N
        if n_recommendations >= len(scores):
            top_indices = np.argsort(scores)[::-1]
        else:
            top_indices = np.argpartition(scores, -n_recommendations)[-n_recommendations:]
            top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]

        # 6. Фильтруем товары с нулевым скором и переводим в ID
        recommendations = []
        for idx in top_indices:
            if scores[idx] > 0:
                recommendations.append(self.idx_to_item_id[idx])
            if len(recommendations) >= n_recommendations:
                break
                
        return recommendations

In [162]:
model_UserKNN = UserKNN(k=5, min_similarity=0.07)
model_UserKNN.fit(df_interactions_train)


In [163]:
data_test_group = df_interactions_test.groupby('user_id')['item_id'].unique().reset_index()
data_test_group.columns = ['user_id', 'item_id']
data_test_group

Exception ignored in: <function tqdm.__del__ at 0x00000237BF03AD40>
Traceback (most recent call last):
  File "c:\Users\kanze\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "c:\Users\kanze\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


,user_id,item_id
0,259,"[10509, 10772]"
1,655,"[11188, 5199]"
2,753,"[9327, 8350]"
3,835,"[5434, 11640, 10878]"
4,960,"[8636, 12770, 6809]"
...,...,...
5014,1096841,"[10219, 3620]"
5015,1097296,"[3006, 16087]"
5016,1097444,"[13650, 12841, 13099, 1124, 12250, 2483]"
5017,1097459,"[11844, 7793]"


In [164]:
# Получаем уникальных пользователей из тестовой выборки
userids = data_test_group['user_id'].values

# Создаем словари для двунаправленного маппинга
user_id_to_idx = {uid: idx for idx, uid in enumerate(userids)}
idx_to_user_id = {idx: uid for uid, idx in user_id_to_idx.items()}

In [165]:
recos_user = []
for i in range(len(userids_test)):
   recommendations = model_UserKNN.recommend(userids[i], n_recommendations=10)
   recos_user.append({'item_id': recommendations})

result_user_knn = pd.DataFrame(recos_user, userids).reset_index()
result_user_knn.to_csv('user_knn_recommendations.csv', index=False, encoding='utf-8')

result_user_knn


,index,item_id
0,259,"[3223, 15947, 13865, 9728, 15521, 9062, 12452,..."
1,655,"[734, 3558, 4151, 16434, 14167, 15297]"
2,753,"[12995, 11987, 3734, 2858, 884, 14998, 4475, 7..."
3,835,"[8254, 11348, 3734, 9696]"
4,960,"[2823, 4495, 3773, 5315, 3384]"
...,...,...
5014,1096841,"[6162, 10878, 12252, 7626, 13865, 211]"
5015,1097296,"[14683, 2720, 4457, 14317, 13865]"
5016,1097444,"[5964, 11469, 281, 4702, 1106, 8419, 9617, 141..."
5017,1097459,"[3153, 6455, 4754, 10440, 9728, 9817]"


In [166]:
# import time
# import pandas as pd
# from tqdm import tqdm

# n_recommendations = 10
# userids = data_test_group['user_id'].values

# recos_user = []
# start_time = time.time()

# # tqdm оборачивает итератор и показывает прогресс-бар
# for i in tqdm(range(len(userids)), desc="Генерация рекомендаций"):
#     try:
#         user_id = userids[i]
#         recommendations = model_UserKNN.recommend(user_id, n_recommendations=n_recommendations)
        
#         recos_user.append({
#             'user_id': user_id,
#             'item_id': recommendations,
#             'n_recommended': len(recommendations)  # Полезно для отладки
#         })
#     except Exception as e:
#         # Логгируем ошибку, но не останавливаем весь процесс
#         recos_user.append({
#             'user_id': userids[i],
#             'item_id': [],
#             'error': str(e)
#         })

# elapsed_time = time.time() - start_time

# # Создаем финальный DataFrame
# result_user_knn = pd.DataFrame(recos_user)

# print(f"✅ Готово! Обработано {len(result_user_knn)} пользователей за {elapsed_time:.2f} сек.")
# print(f"⚡ Средняя скорость: {len(result_user_knn) / elapsed_time:.1f} пользователей/сек")

In [167]:
print('UserKNN_map_k =', map_k(result_user_knn['item_id'], data_test_group['item_id'], k=10, u=len(data_test_group)))

UserKNN_map_k = 0.007444101303933947


ItemKNN

In [168]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

class ItemKNN:
    def __init__(self, k=10, min_similarity=0.1):
        self.k = k
        self.min_similarity = min_similarity
        
        # Словари для быстрого маппинга ID <-> Индекс (O(1))
        self.user_id_to_idx = {}
        self.idx_to_user_id = {}
        self.item_id_to_idx = {}
        self.idx_to_item_id = {}
        
        self.user_item_matrix = None  # Разреженная матрица (CSR): Users x Items
        self.item_similarity = None   # Матрица схожести товаров: Items x Items
        self.n_users = 0
        self.n_items = 0

    def fit(self, interactions):
        """Обучение модели: построение матриц и расчет схожести товаров."""
        self._build_mappings(interactions)
        self._build_matrix(interactions)
        self._compute_similarity()
        return self

    def _build_mappings(self, interactions):
        """Создает словари для быстрого перевода ID в индексы."""
        unique_users = interactions['user_id'].unique()
        unique_items = interactions['item_id'].unique()
        
        self.user_id_to_idx = {uid: idx for idx, uid in enumerate(unique_users)}
        self.idx_to_user_id = {idx: uid for uid, idx in self.user_id_to_idx.items()}
        
        self.item_id_to_idx = {iid: idx for idx, iid in enumerate(unique_items)}
        self.idx_to_item_id = {idx: iid for iid, idx in self.item_id_to_idx.items()}
        
        self.n_users = len(unique_users)
        self.n_items = len(unique_items)

    def _build_matrix(self, interactions):
        """Создает разреженную матрицу пользователь-товар."""
        # Используем map вместо повторного создания Categorical
        user_codes = interactions['user_id'].map(self.user_id_to_idx)
        item_codes = interactions['item_id'].map(self.item_id_to_idx)
        
        # Фильтруем строки с NaN (если вдруг есть новые ID)
        mask = user_codes.notna() & item_codes.notna()
        
        self.user_item_matrix = csr_matrix(
            (interactions.loc[mask, 'watched_pct'], 
             (user_codes.loc[mask], item_codes.loc[mask])),
            shape=(self.n_users, self.n_items)
        )

    def _compute_similarity(self):
        """Вычисляет матрицу косинусной схожести между ТОВАРАМИ."""
        # Транспонируем: (Items x Users) для расчета схожести колонок
        # cosine_similarity работает эффективно с разреженными матрицами на входе
        self.item_similarity = cosine_similarity(
            self.user_item_matrix.T, 
            dense_output=True  # Результат все равно будет плотным (Items x Items)
        )
        np.fill_diagonal(self.item_similarity, 0)  # Товар не похож сам на себя

    def _get_user_idx(self, user_id):
        return self.user_id_to_idx.get(user_id)

    def _get_item_idx(self, item_id):
        return self.item_id_to_idx.get(item_id)

    def _get_similar_items(self, item_idx):
        """Возвращает индексы и веса похожих товаров для данного индекса."""
        similarities = self.item_similarity[item_idx]
        
        # Берем топ-k + фильтр по порогу
        # Используем argsort для сортировки по убыванию
        top_indices = np.argsort(similarities)[::-1][:self.k]
        
        similar_items = []
        for idx in top_indices:
            sim = similarities[idx]
            if sim >= self.min_similarity:
                similar_items.append((idx, sim))
            else:
                break  # Отсортировано по убыванию, дальше нет смысла
        return similar_items

    def predict(self, user_id, item_id):
        """Предсказывает рейтинг для пары пользователь-товар."""
        user_idx = self._get_user_idx(user_id)
        target_item_idx = self._get_item_idx(item_id)
        
        if user_idx is None or target_item_idx is None:
            return 0.0

        similar_items = self._get_similar_items(target_item_idx)
        if not similar_items:
            return 0.0

        # Векторизованный сбор рейтингов пользователя для похожих товаров
        neighbor_indices = [idx for idx, _ in similar_items]
        neighbor_weights = np.array([w for _, w in similar_items])
        
        # Получаем рейтинг пользователя для этих товаров из разреженной матрицы
        # Берем срез строки пользователя по нужным колонкам (товарам)
        user_ratings_row = self.user_item_matrix[user_idx, neighbor_indices]
        neighbor_ratings = np.asarray(user_ratings_row).flatten()
        
        # Фильтруем товары, которые пользователь еще не смотрел (рейтинг = 0)
        mask = neighbor_ratings > 0
        if not np.any(mask):
            return 0.0
            
        weights = neighbor_weights[mask]
        ratings = neighbor_ratings[mask]
        
        if np.sum(weights) > 0:
            return float(np.average(ratings, weights=weights))
        else:
            return float(np.mean(ratings))

    def recommend(self, user_id, n_recommendations=10):
        """Генерирует топ-N рекомендаций (Векторизовано)."""
        user_idx = self._get_user_idx(user_id)
        if user_idx is None:
            return []

        # 1. Получаем товары, которые пользователь уже смотрел
        user_rated_items = self.user_item_matrix[user_idx].indices
        if len(user_rated_items) == 0:
            return []  # Холодный старт: нет истории -> нет рекомендаций

        # 2. Для каждого просмотренного товара находим похожие и агрегируем скоры
        # Создаем массив для накопления скоров всех товаров
        scores = np.zeros(self.n_items)
        
        for rated_item_idx in user_rated_items:
            similar_items = self._get_similar_items(rated_item_idx)
            if not similar_items:
                continue
                
            # Веса схожести для похожих товаров
            for sim_idx, sim_weight in similar_items:
                # Если товар еще не был просмотрен пользователем, добавляем вклад в скор
                if sim_idx not in user_rated_items:
                    # Рейтинг пользователя для просмотренного товара как вес влияния
                    user_rating = self.user_item_matrix[user_idx, rated_item_idx]
                    scores[sim_idx] += sim_weight * user_rating

        # 3. Обнуляем уже просмотренные товары (защита от дубликатов)
        scores[user_rated_items] = -np.inf

        # 4. Находим топ-N индексов с максимальным скором
        # argpartition быстрее, чем полный argsort
        if n_recommendations >= len(scores):
            top_indices = np.argsort(scores)[::-1]
        else:
            top_indices = np.argpartition(scores, -n_recommendations)[-n_recommendations:]
            # Сортируем только отобранный топ для корректного порядка
            top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]

        # 5. Переводим индексы обратно в ID товаров
        recommendations = []
        for idx in top_indices:
            if scores[idx] > 0:  # Берем только товары с положительным скором
                recommendations.append(self.idx_to_item_id[idx])
            if len(recommendations) >= n_recommendations:
                break
                
        return recommendations

In [169]:
model_ItemKNN = ItemKNN(k=9, min_similarity=0.1)
model_ItemKNN.fit(df_interactions_train)

In [170]:
recos_item = []
for i in range(len(userids_test)):
   recommendations = model_ItemKNN.recommend(userids[i], n_recommendations=10)
   recos_item.append({'item_id': recommendations})

result_item_knn = pd.DataFrame(recos_item, userids).reset_index()
result_item_knn

,index,item_id
0,259,"[4034, 3168, 11840, 15521, 12838, 14135, 7412,..."
1,655,"[10645, 1839, 835, 16349, 11298, 16434, 11200,..."
2,753,"[928, 1757, 2257, 7442, 3708, 14591, 14728, 13..."
3,835,"[7005, 5532, 3471, 15191, 14179, 13683, 11291,..."
4,960,"[7442, 16238, 3791, 7485, 5093, 12428, 14733, ..."
...,...,...
5014,1096841,"[11340, 9853, 5061, 11576, 14456, 12829, 12514..."
5015,1097296,"[12278, 7027, 7189, 1351, 6666, 6576, 10506, 1..."
5016,1097444,"[5964, 9199, 1757, 5114, 2590, 254, 10427, 187..."
5017,1097459,"[15861, 7437, 14222, 7755, 14223, 548, 11135, ..."


In [171]:
print('ItemKNN_map_k =', map_k(result_item_knn['item_id'], data_test_group['item_id'], k=10, u=len(data_test_group)))

ItemKNN_map_k = 0.0028397644506431116
